# 04 — Förbered Epic-datasetet
Söker igenom oanvända CelebA-mustaschbilder med en **befintlig** epic-modell för att hitta nya epic/thin-kandidater att lägga till manuellt i `data/epic_dataset/`.

**OBS:** `IMG_SIZE` måste matcha den modell du laddar — gamla modeller (128x128) vs nya (178x178).

In [1]:
import os
import shutil
import glob
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array

EPIC_MODEL_PATH = 'models/epic_detector_3.keras'
SOURCE_DIR      = 'data/img/img_align_celeba'
ATTR_PATH       = 'data/list_attr_celeba.csv'
IMG_SIZE        = (178, 178)  # matcha modellen ovan!

REVIEW_EPIC_DIR = 'data/review_epic_sort/epic_high_score'
REVIEW_THIN_DIR = 'data/review_epic_sort/thin_low_score'
os.makedirs(REVIEW_EPIC_DIR, exist_ok=True)
os.makedirs(REVIEW_THIN_DIR, exist_ok=True)

epic_model = load_model(EPIC_MODEL_PATH)
attrs      = pd.read_csv(ATTR_PATH)

print('Modell laddad! Input shape:', epic_model.input_shape)

Modell laddad! Input shape: (None, 178, 178, 3)


## Hitta oanvända mustasch-bilder

In [2]:
mustache_pool = attrs[attrs['Mustache'] == 1].copy()
print('CelebA mustache-bilder totalt:', len(mustache_pool))

already_used = set()
for folder in [
    'data/epic_dataset/epic',
    'data/epic_dataset/thin',
]:
    if os.path.exists(folder):
        already_used |= set(os.listdir(folder))

mustache_pool = mustache_pool[~mustache_pool['image_id'].isin(already_used)]
print('Kvar att söka i:', len(mustache_pool))

CelebA mustache-bilder totalt: 8417
Kvar att söka i: 8409


## Scora alla oanvända bilder med nuvarande epic-modell

In [ ]:
rows = []

for i, fname in enumerate(mustache_pool['image_id']):
    path = os.path.join(SOURCE_DIR, fname)
    if not os.path.exists(path):
        continue

    img = load_img(path, target_size=IMG_SIZE)
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)

    score = float(epic_model.predict(arr, verbose=0)[0][0])

    rows.append({'filename': fname, 'path': path, 'epic_score': score})

    if (i + 1) % 500 == 0:
        print(f'{i+1}/{len(mustache_pool)} klara')

epic_search_df = pd.DataFrame(rows)
print('Klart!')

## Kopiera topp-kandidater till review-mappar
Granska manuellt i Finder/Filutforskare innan du flyttar dem till `data/epic_dataset/`.

In [ ]:
EPIC_TOP_N = 2000
THIN_TOP_N = 2000

epic_candidates = epic_search_df.sort_values('epic_score', ascending=False).head(EPIC_TOP_N)
thin_candidates = epic_search_df.sort_values('epic_score', ascending=True).head(THIN_TOP_N)

for _, row in epic_candidates.iterrows():
    src = row['path']
    dst = os.path.join(REVIEW_EPIC_DIR, f"{row['epic_score']:.3f}_{row['filename']}")
    if os.path.exists(src):
        shutil.copy2(src, dst)

for _, row in thin_candidates.iterrows():
    src = row['path']
    dst = os.path.join(REVIEW_THIN_DIR, f"{row['epic_score']:.3f}_{row['filename']}")
    if os.path.exists(src):
        shutil.copy2(src, dst)

print(f'Kopierade {len(epic_candidates)} high-score bilder till {REVIEW_EPIC_DIR}')
print(f'Kopierade {len(thin_candidates)} low-score bilder till {REVIEW_THIN_DIR}')

## Klart!
Granska bilderna i `data/review_epic_sort/` manuellt och flytta de som ser rätt ut till `data/epic_dataset/epic` eller `data/epic_dataset/thin`.

Gå sedan till **05_train_epic_model.ipynb**.